# Food Detection Training - YOLO11

Notebook reproductible pour entrainer un detecteur d'objets food sur 4 classes: pizza, sandwich, pasta, hotdog.

Architecture retenue:
- baseline: YOLO11s
- modele principal: YOLO11m

Le choix est du transfer learning: on part de poids pre-entraines COCO puis on fine-tune sur le dataset food.

## 1. Setup

Installe `ultralytics` si necessaire, importe les librairies, fixe les seeds, et verifie le GPU.

In [ ]:
# Ne lance pas un pip install manuel dans ce notebook.
# Si l'environnement n'est pas pret, execute dans un terminal Jupyter:
# bash scripts/setup_foodex_env.sh

import sys
if sys.version_info < (3, 10):
    raise RuntimeError(
        f'Kernel Python {sys.version_info.major}.{sys.version_info.minor} non supporte. '
        'Utilise un kernel Python >= 3.10 avec PyTorch CUDA fonctionnel.'
    )


from pathlib import Path
import json
import random
import shutil
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from ultralytics import YOLO, settings

warnings.filterwarnings('ignore')
settings.update({'mlflow': False})
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'configs' / 'food_dataset.yaml').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Impossible de trouver la racine du depot foodex. Lance le notebook depuis foodex/ ou foodex/notebooks/.')

PROJECT_ROOT = find_project_root()
DATA_YAML = PROJECT_ROOT / 'configs' / 'food_dataset.yaml'
DATASET_DIR = PROJECT_ROOT / 'dataset'
RUNS_DIR = PROJECT_ROOT / 'runs_food'
REPORT_FIGURES_DIR = PROJECT_ROOT / 'report' / 'figures'
REPORT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Dataset yaml:', DATA_YAML)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Dataset Audit

Cette partie verifie les erreurs frequentes: labels manquants, images sans labels, fichiers parasites, classes invalides, bboxes hors limites.

In [ ]:
with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg['names']
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

splits = ['train', 'val', 'test']
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(split):
    base = DATASET_DIR / 'images' / split
    return sorted([p for p in base.rglob('*') if p.suffix.lower() in image_exts and ':Zone.Identifier' not in p.name])

def label_path_for_image(image_path):
    rel = image_path.relative_to(DATASET_DIR / 'images')
    return DATASET_DIR / 'labels' / rel.with_suffix('.txt')

def parse_label_file(label_path):
    rows = []
    if not label_path.exists():
        return rows
    for line_no, line in enumerate(label_path.read_text().splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            rows.append({'error': 'malformed', 'line_no': line_no, 'raw': line})
            continue
        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
            rows.append({'class_id': cls, 'x': x, 'y': y, 'w': w, 'h': h, 'line_no': line_no})
        except ValueError:
            rows.append({'error': 'parse_error', 'line_no': line_no, 'raw': line})
    return rows

audit_rows = []
invalid_rows = []
missing_labels = []

for split in splits:
    for image_path in list_images(split):
        cls_folder = image_path.parent.name
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            missing_labels.append(str(image_path))
            continue
        parsed = parse_label_file(label_path)
        for obj in parsed:
            if 'error' in obj:
                invalid_rows.append({'split': split, 'label': str(label_path), **obj})
                continue
            valid_class = 0 <= obj['class_id'] < len(class_names)
            valid_bbox = all(0 <= obj[k] <= 1 for k in ['x', 'y', 'w', 'h']) and obj['w'] > 0 and obj['h'] > 0
            if not valid_class or not valid_bbox:
                invalid_rows.append({'split': split, 'label': str(label_path), 'error': 'invalid_class_or_bbox', **obj})
            audit_rows.append({
                'split': split,
                'image': str(image_path),
                'label': str(label_path),
                'folder_class': cls_folder,
                'class_id': obj['class_id'],
                'class_name': class_names[obj['class_id']] if 0 <= obj['class_id'] < len(class_names) else 'invalid',
                'bbox_area': obj['w'] * obj['h'],
                'bbox_w': obj['w'],
                'bbox_h': obj['h'],
            })

audit_df = pd.DataFrame(audit_rows)
zone_files = sorted(DATASET_DIR.rglob('*:Zone.Identifier*'))

summary = {
    'num_images': {split: len(list_images(split)) for split in splits},
    'num_annotations': int(len(audit_df)),
    'missing_labels': len(missing_labels),
    'invalid_label_rows': len(invalid_rows),
    'zone_identifier_files': len(zone_files),
}

print(json.dumps(summary, indent=2))
display(audit_df.head())
if missing_labels:
    print('Images sans label, exemples:')
    print('\n'.join(missing_labels[:10]))
if invalid_rows:
    print('Labels invalides, exemples:')
    display(pd.DataFrame(invalid_rows).head(10))

In [ ]:
class_counts = audit_df.groupby(['split', 'class_name']).size().reset_index(name='count')
display(class_counts)

plt.figure(figsize=(9, 5))
sns.barplot(data=class_counts, x='class_name', y='count', hue='split')
plt.title('Class distribution by split')
plt.xlabel('Class')
plt.ylabel('Number of annotated objects')
plt.tight_layout()
plt.savefig(REPORT_FIGURES_DIR / 'class_distribution.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(data=audit_df, x='bbox_area', hue='class_name', bins=40, kde=False)
plt.title('Bounding-box area distribution')
plt.xlabel('Normalized bbox area')
plt.tight_layout()
plt.savefig(REPORT_FIGURES_DIR / 'bbox_area_distribution.png', dpi=200)
plt.show()

## 3. Visualisation des annotations

On affiche des exemples avec bounding boxes pour verifier visuellement la qualite des labels.

In [ ]:
def draw_yolo_boxes(image_path, ax=None):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    label_path = label_path_for_image(image_path)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)
    ax.axis('off')
    for obj in parse_label_file(label_path):
        if 'error' in obj:
            continue
        cls, xc, yc, bw, bh = obj['class_id'], obj['x'], obj['y'], obj['w'], obj['h']
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        rect = plt.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2, color='lime')
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 4, 0), class_names[cls], color='black', fontsize=9, bbox=dict(facecolor='lime', alpha=0.8, pad=1))
    ax.set_title(image_path.parent.name + '/' + image_path.name, fontsize=9)

sample_images = random.sample(list_images('train'), min(9, len(list_images('train'))))
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, img in zip(axes.ravel(), sample_images):
    draw_yolo_boxes(img, ax=ax)
plt.tight_layout()
plt.savefig(REPORT_FIGURES_DIR / 'annotation_examples.png', dpi=200)
plt.show()

## 4. Configuration d'entrainement

Hyperparametres proposes pour une RTX A6000:
- `imgsz=640`: standard YOLO, stable pour commencer.
- `epochs=50` + `patience=10`: premier run fiable et assez court pour valider la pipeline.
- `batch=16`: batch fixe pour eviter les lenteurs/autobatch et reduire la pression memoire/disque.
- `workers=0`: evite les blocages DataLoader multiprocessing observes avec Python 3.13/Jupyter.
- `optimizer='AdamW'`: robuste pour fine-tuning avec weight decay.
- `cos_lr=True`: scheduler cosine pour une fin d'entrainement plus stable.
- `mosaic=1.0`, `close_mosaic=10`: augmentation utile au debut, desactivee en fin d'entrainement.

In [ ]:
TRAIN_KWARGS = dict(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=0,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=3,
    cos_lr=True,
    seed=SEED,
    device=0,
    project=str(RUNS_DIR),
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.05,
    close_mosaic=10,
    plots=True,
    save=True,
    verbose=True,
)

TRAIN_KWARGS

## 5. Baseline YOLO11s

Lance cette cellule pour entrainer la baseline. Elle sert a comparer avec YOLO11m et a detecter rapidement un probleme de dataset.

In [ ]:
RUN_BASELINE = False  # Mets True pour lancer l'entrainement baseline.

if RUN_BASELINE:
    baseline_model = YOLO('yolo11s.pt')
    baseline_results = baseline_model.train(name='yolo11s_baseline_safe', **TRAIN_KWARGS)
else:
    print('Baseline non lancee. Mets RUN_BASELINE=True pour entrainer YOLO11s.')

## 6. Modele principal YOLO11m

Lance cette cellule sur la A6000. Le meilleur modele sera dans `runs_food/yolo11m_food/weights/best.pt`.

In [ ]:
RUN_MAIN_TRAINING = False  # Mets True pour lancer l'entrainement principal.

if RUN_MAIN_TRAINING:
    main_model = YOLO('yolo11m.pt')
    main_results = main_model.train(name='yolo11m_food', **TRAIN_KWARGS)
else:
    print('Training principal non lance. Mets RUN_MAIN_TRAINING=True pour entrainer YOLO11m.')

## 7. Analyse des courbes d'entrainement

Cette cellule lit les fichiers `results.csv` generes par Ultralytics et trace les courbes principales.

In [ ]:
def load_results(run_name):
    path = RUNS_DIR / run_name / 'results.csv'
    if not path.exists():
        print(f'Results not found: {path}')
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

def plot_training_curves(run_names):
    metrics = [
        'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
        'metrics/precision(B)', 'metrics/recall(B)',
        'metrics/mAP50(B)', 'metrics/mAP50-95(B)'
    ]
    available = []
    dfs = {}
    for run in run_names:
        df = load_results(run)
        if df is not None:
            dfs[run] = df
            available.extend([m for m in metrics if m in df.columns])
    available = list(dict.fromkeys(available))
    if not dfs:
        return
    ncols = 2
    nrows = int(np.ceil(len(available) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, metric in zip(axes, available):
        for run, df in dfs.items():
            ax.plot(df['epoch'], df[metric], label=run)
        ax.set_title(metric)
        ax.set_xlabel('Epoch')
        ax.legend()
    for ax in axes[len(available):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(REPORT_FIGURES_DIR / 'training_curves.png', dpi=200)
    plt.show()

plot_training_curves(['yolo11s_baseline_safe', 'yolo11m_food'])

## 8. Evaluation test

Charge le meilleur poids disponible et evalue sur le split test.

In [ ]:
def best_weight(run_name):
    path = RUNS_DIR / run_name / 'weights' / 'best.pt'
    return path if path.exists() else None

candidate_runs = ['yolo11m_food', 'yolo11s_baseline_safe']
weights = [best_weight(run) for run in candidate_runs]
weights = [w for w in weights if w is not None]

if weights:
    selected_weight = weights[0]
    print('Selected weight:', selected_weight)
    eval_model = YOLO(str(selected_weight))
    test_metrics = eval_model.val(data=str(DATA_YAML), split='test', project=str(RUNS_DIR), name='test_eval', plots=True)
    print(test_metrics)
    test_eval_dir = RUNS_DIR / 'test_eval'
    for src_name, dst_name in [
        ('confusion_matrix.png', 'confusion_matrix.png'),
        ('confusion_matrix_normalized.png', 'confusion_matrix_normalized.png'),
        ('PR_curve.png', 'pr_curve.png'),
        ('F1_curve.png', 'f1_curve.png'),
    ]:
        src = test_eval_dir / src_name
        if src.exists():
            shutil.copy(src, REPORT_FIGURES_DIR / dst_name)
else:
    print('Aucun poids best.pt trouve. Lance un entrainement avant cette cellule.')

## 9. Inference visuelle

Cette partie lance des predictions sur des images test et sauvegarde une figure pour le rapport.

In [ ]:
def plot_predictions(model, images, conf=0.25, max_det=20, save_path=None):
    n = len(images)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = np.array(axes).reshape(-1)
    results = model.predict(source=[str(p) for p in images], conf=conf, max_det=max_det, verbose=False)
    for ax, result, image_path in zip(axes, results, images):
        plotted = result.plot()[:, :, ::-1]
        ax.imshow(plotted)
        ax.set_title(image_path.name, fontsize=9)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
    plt.show()

if weights:
    inference_model = YOLO(str(weights[0]))
    test_images = list_images('test')
    sample_test_images = random.sample(test_images, min(9, len(test_images)))
    plot_predictions(inference_model, sample_test_images, conf=0.25, save_path=REPORT_FIGURES_DIR / 'inference_examples.png')
else:
    print('Aucun poids disponible pour inference.')

## 10. Export du modele

Export optionnel en ONNX pour deploiement. Garde toujours aussi le `best.pt`.

In [ ]:
EXPORT_ONNX = False

if EXPORT_ONNX and weights:
    export_model = YOLO(str(weights[0]))
    export_path = export_model.export(format='onnx', imgsz=640, simplify=True)
    print('Exported:', export_path)
else:
    print('Export ONNX non lance. Mets EXPORT_ONNX=True apres entrainement si besoin.')

## 11. Recommandations apres resultats

Regles de decision:

- Overfitting: train loss baisse, val mAP stagne ou baisse. Actions: reduire modele vers YOLO11s, ajouter data, augmenter regularisation, verifier duplicats.
- Underfitting: train et val faibles. Actions: augmenter epochs, passer `imgsz=768`, tester YOLO11l, verifier labels.
- Faible recall sur une classe: collecter plus d'images de cette classe et ajouter des hard examples.
- Confusion entre classes: inspecter matrice de confusion, ajouter exemples proches et labels plus coherents.
- Petits objets rates: augmenter `imgsz` a 768 ou 896, ou verifier les annotations petites.